# 04 : Assistant GenAI - text-to-SQL

**Entrée** : base `data/openpayments.sqlite`
**LLM** : local via Ollama (llama3.2), configurable (OpenAI possible)
**Prérequis** : application **Ollama** en marche (ou une clé OpenAI)

---

### Objectif

Permettre à un interlocuteur métier (marketing, vente) d'interroger les données en langage naturel, sans écrire de SQL. L'assistant traduit la question en requête SQL, l'exécute sur la base, puis rédige la réponse.

### Fonctionnement

1. **Vue sémantique** `paiements` : renomme les colonnes CMS très longues (ex. `applicable_manufacturer_or_applicable_gpo_making_payment_name`) en noms courts et intuitifs (`laboratoire`, `montant`, `nature`...), pour aider le petit modèle malgré sa taille réduite.
2. **Génération SQL** : le LLM traduit la question en requête SQL à partir du schéma de la vue.
3. **Garde-fous** (`is_safe`) : autorise uniquement un `SELECT`, rejette toute requête contenant un `;` (empêche l'enchaînement de plusieurs instructions), bloque explicitement `DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `PRAGMA`. En cas d'échec d'exécution, une nouvelle tentative est faite en renvoyant l'erreur au LLM pour qu'il corrige.
4. **Narration** : un second appel LLM reformule le résultat SQL en 1 à 3 phrases.

### Limite assumée

Un petit modèle local (3 milliards de paramètres) produit du SQL correct sur des questions simples, mais reste imparfait sur les formulations complexes. Arbitrage assumé **coût / fiabilité** : gratuit et 100 % local, contre un modèle plus capable mais payant via API (configurable via `LLM_PROVIDER`).

### Démonstration (2 questions)

**Question simple (comptage)** : *« Combien de professionnels de santé distincts en 2023 ? »*

```sql
SELECT COUNT(DISTINCT professionnel_id) FROM paiements WHERE annee = 2023

## 0. Configuration (schéma + prompts)

In [1]:
import os, re, sqlite3, pathlib
import pandas as pd
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

os.environ.setdefault("LLM_PROVIDER", "ollama")
os.environ.setdefault("OLLAMA_MODEL", "llama3.2")

ROOT = pathlib.Path.cwd().parent if (pathlib.Path.cwd().parent / "data").exists() else pathlib.Path.cwd()
DB_PATH = ROOT / "data" / "openpayments.sqlite"

SCHEMA = """
Table `paiements` (un paiement d'un laboratoire a un professionnel de sante) :
- professionnel_id (TEXT) : identifiant du professionnel de sante
- etat (TEXT)             : etat (ex. 'WV')
- specialite (TEXT)       : specialite du professionnel
- laboratoire (TEXT)      : nom du laboratoire payeur
- montant (REAL)          : montant du paiement en USD
- nature (TEXT)           : nature ('Food and Beverage', 'Travel and Lodging', 'Consulting Fee', ...)
- annee (INTEGER)         : 2022 ou 2023
"""

SQL_PROMPT = (
    "Tu traduis une question en une requete SQL SQLite.\n{schema}\n"
    "Regles STRICTES : uniquement un SELECT ; une seule instruction sans point-virgule ;\n"
    "table `paiements` uniquement ; LIMIT 20 max si liste ; reponds avec la requete SEULE.\n\n"
    "Question : {question}\nSQL :"
)
NARRATE_PROMPT = (
    "Tu es analyste de donnees commerciales pharma. En 1 a 3 phrases claires,\n"
    "reponds a la question a partir du resultat SQL.\n\n"
    "Question : {question}\nResultat :\n{result}\n\nReponse :"
)

## 1. Vue sémantique, garde-fous et chaîne

On crée la vue `paiements`, puis les fonctions : `get_llm` (LLM configurable), `extract_sql` (récupère la requête même enrobée), `is_safe` (bloque tout ce qui n'est pas un SELECT), et `ask` (génère le SQL, l'exécute avec une nouvelle tentative en cas d'erreur, puis rédige la réponse).

In [2]:
def get_llm(temperature=0.0):
    from langchain_ollama import ChatOllama
    return ChatOllama(model=os.getenv("OLLAMA_MODEL", "llama3.2"), temperature=temperature)

def ensure_view():
    with sqlite3.connect(DB_PATH) as con:
        con.execute("""
        CREATE VIEW IF NOT EXISTS paiements AS
        SELECT covered_recipient_profile_id AS professionnel_id, recipient_state AS etat,
               specialty AS specialite,
               applicable_manufacturer_or_applicable_gpo_making_payment_name AS laboratoire,
               total_amount_of_payment_usdollars AS montant,
               nature_of_payment_or_transfer_of_value AS nature, program_year AS annee
        FROM payments""")

def extract_sql(text):
    text = re.sub(r"```(?:sql)?", "", text, flags=re.IGNORECASE).strip("` \n")
    m = re.search(r"(?is)\bselect\b.*", text)
    sql = m.group(0) if m else text
    # coupe au premier marqueur de fin de requete (le LLM ajoute parfois une
    # explication apres le SQL, dont un ';' qui ferait echouer le garde-fou is_safe)
    sql = re.split(r";|```|\n\s*\n", sql)[0]
    return sql.strip()

def is_safe(sql):
    low = " " + sql.lower().strip() + " "
    if not low.strip().startswith("select") or ";" in sql:
        return False
    return not any(k in low for k in ("drop ", "delete ", "update ", "insert ", "alter ", "pragma"))

def run_sql(sql):
    with sqlite3.connect(DB_PATH) as con:
        return pd.read_sql(sql, con)

def ask(question, max_retries=1, verbose=True):
    ensure_view()
    llm = get_llm()
    sql_chain = ChatPromptTemplate.from_template(SQL_PROMPT) | llm | StrOutputParser()
    error, sql, df = None, "", None
    for _ in range(max_retries + 1):
        q = question if error is None else f"{question}\n(Requete precedente en echec: {error}. Corrige.)"
        sql = extract_sql(sql_chain.invoke({"schema": SCHEMA, "question": q}))
        if not is_safe(sql):
            return "Requete refusee (SELECT uniquement).", sql, None
        try:
            df = run_sql(sql); error = None; break
        except Exception as exc:
            error = str(exc)
    if error is not None:
        return f"Echec SQL : {error}", sql, None
    narrate = ChatPromptTemplate.from_template(NARRATE_PROMPT) | llm | StrOutputParser()
    answer = narrate.invoke({"question": question, "result": df.head(20).to_string(index=False)})
    if verbose:
        print("SQL:", sql, "\n"); print(df.head(10).to_string(index=False))
    return answer, sql, df

## 2. Démonstration

Deux questions métier. La première (comptage simple) est bien gérée ; la seconde (agrégation + tri) montre les limites d'un petit modèle.

In [3]:
answer, sql, df = ask("Combien de professionnels de sante distincts en 2023 ?")
print("\nREPONSE:", answer)

SQL: SELECT COUNT(DISTINCT professionnel_id) FROM paiements WHERE annee = 2023 

 COUNT(DISTINCT professionnel_id)
                             6230

REPONSE: Il y a 6230 professionnels de santé distincts en 2023.


In [4]:
answer, sql, df = ask("Quels sont les 5 laboratoires qui depensent le plus au total ?")
print("\nREPONSE:", answer)

SQL: SELECT laboratoire 
FROM paiements 
GROUP BY laboratoire 
ORDER BY SUM(montant) DESC 
LIMIT 5 

                 laboratoire
                 ABBVIE INC.
         ABBOTT LABORATORIES
             MEDTRONIC, INC.
JANSSEN PHARMACEUTICALS, INC
NEUROCRINE BIOSCIENCES, INC.

REPONSE: Les 5 laboratoires qui dépensent le plus au total sont :

1. ABBVIE INC. (données non disponibles)
2. ABBOTT LABORATORIES (données non disponibles)
3. MEDTRONIC, INC.
4. JANSSEN PHARMACEUTICALS, INC
5. NEUROCRINE BIOSCIENCES, INC


> **Observations.** Sur une question simple (comptage), le SQL généré est correct et la réponse juste (~6 230 HCP en 2023). Sur l’agrégation, le petit modèle construit une requête SQL valide (GROUP BY + ORDER BY SUM(montant) DESC) mais **oublie d’inclure le montant agrégé dans le SELECT** : la table ne renvoie que le nom du laboratoire, sans son total, et le LLM narre alors une réponse partielle (« données non disponibles »). Les **garde-fous** sécurisent l’exécution (aucune requête destructrice ne peut passer) mais ne corrigent pas les erreurs de raisonnement du modèle ; un modèle plus capable (via API) réduirait cette limite.

## 📝 Synthèse de l'Assistant GenAI

Ce notebook a permis de construire et d'évaluer un assistant text-to-SQL permettant d'interroger la base de paiements en langage naturel, sans écrire de requête SQL.

1. **Architecture mise en place**
   * Vue sémantique aux noms de colonnes courts, qui masque la complexité du schéma CMS brut au modèle.
   * Garde-fous stricts : SELECT uniquement, blocage des mots-clés destructeurs (`DROP`, `DELETE`, `UPDATE`, `INSERT`, `ALTER`, `PRAGMA`), une seule instruction par requête.
   * Boucle génération SQL → exécution → nouvelle tentative en cas d'erreur, puis narration de la réponse en langage naturel.

2. **Résultats de la démonstration**
   * Question simple (comptage) : SQL correct, réponse juste (**6 230** professionnels distincts en 2023).
   * Question complexe (agrégation + tri) : requête valide (bon `GROUP BY` / `ORDER BY`) mais incomplète : le montant agrégé est omis du `SELECT`, d'où une réponse partielle du modèle.

3. **Enseignement et décision**
   * Un modèle local de 3 milliards de paramètres (llama3.2) suffit pour des requêtes simples, mais montre ses limites dès qu'il faut articuler plusieurs opérations SQL. Les garde-fous protègent contre les requêtes dangereuses, mais **ne garantissent pas l'exactitude du résultat métier**.
   * *Décision* : pour un usage en production, prévoir soit un modèle plus capable via API, soit une validation humaine systématique des réponses avant diffusion aux équipes commerciales.
